# google_cloud_audit_log smoke test

Smoke test for `google-cloud-audit-log` — the protobuf-only distribution of the
Cloud Audit Logging schema. Its import name is `google.cloud.audit` (NOT
`google.cloud.audit_log`). All examples construct protos offline.

In [ ]:
import pkgutil, importlib
import google.cloud.audit as _pkg_under_test  # import name for the google-cloud-audit-log dist

failed = []
for module_info in pkgutil.walk_packages(_pkg_under_test.__path__, prefix=f"{_pkg_under_test.__name__}."):
    try:
        importlib.import_module(module_info.name)
    except Exception as e:
        failed.append((module_info.name, type(e).__name__, str(e)))

if failed:
    for name, etype, msg in failed:
        print(f"FAIL {name}: {etype}: {msg}")
    raise AssertionError(f"{len(failed)} submodule(s) of {_pkg_under_test.__name__} failed to import")
print(f"OK — all submodules of {_pkg_under_test.__name__} imported cleanly")

In [ ]:
# Example 1: Construct an AuditLog protobuf message and set core fields.
from google.cloud.audit import audit_log_pb2

log = audit_log_pb2.AuditLog(
    service_name="storage.googleapis.com",
    method_name="storage.objects.get",
    resource_name="projects/_/buckets/demo-bucket",
)
log.num_response_items = 1
print(log)
assert log.service_name == "storage.googleapis.com"
assert log.method_name == "storage.objects.get"
assert log.num_response_items == 1

In [ ]:
# Example 2: Nested audit fields — authentication, authorization, request Struct.
from google.cloud.audit import audit_log_pb2

log = audit_log_pb2.AuditLog(service_name="bigquery.googleapis.com")
log.authentication_info.principal_email = "svc@example.iam.gserviceaccount.com"
authz = log.authorization_info.add()
authz.permission = "bigquery.jobs.create"
authz.granted = True
log.request["query"] = "SELECT 1"

print(f"principal: {log.authentication_info.principal_email}")
print(f"authz: {authz.permission} granted={authz.granted}")
print(f"request struct: {dict(log.request)}")
assert len(log.authorization_info) == 1
assert log.request["query"] == "SELECT 1"

In [ ]:
# Example 3: Wire-format round trip — SerializeToString / FromString.
from google.cloud.audit import audit_log_pb2

log = audit_log_pb2.AuditLog(
    service_name="compute.googleapis.com",
    method_name="v1.compute.instances.insert",
)
log.authentication_info.principal_email = "user@example.com"

data = log.SerializeToString()
restored = audit_log_pb2.AuditLog.FromString(data)
print(f"serialized {len(data)} bytes")
print(f"restored principal: {restored.authentication_info.principal_email}")
assert restored == log
assert restored.service_name == "compute.googleapis.com"

In [ ]:
# Example 4: Descriptor introspection — the proto schema the package ships.
from google.cloud.audit import audit_log_pb2

desc = audit_log_pb2.AuditLog.DESCRIPTOR
print(f"full_name: {desc.full_name}")
field_names = [f.name for f in desc.fields]
print(f"{len(field_names)} fields: {field_names}")
assert desc.full_name == "google.cloud.audit.AuditLog"
for expected in ("service_name", "method_name", "status", "request_metadata"):
    assert expected in field_names, f"missing field {expected}"
print("AuditLog schema exposes the documented core fields")

In [ ]:
# Example 5: The sibling BigQueryAuditMetadata message also ships in this dist.
from google.cloud.audit import bigquery_audit_metadata_pb2

meta = bigquery_audit_metadata_pb2.BigQueryAuditMetadata()
print(f"full_name: {meta.DESCRIPTOR.full_name}")
assert meta.DESCRIPTOR.full_name == "google.cloud.audit.BigQueryAuditMetadata"

data = meta.SerializeToString()
print(f"empty message serializes to {len(data)} bytes")
assert bigquery_audit_metadata_pb2.BigQueryAuditMetadata.FromString(data) == meta
print("BigQueryAuditMetadata round trip OK")